# OBCB Model: Verification of PN, PS, PNS Values

Verifies every number quoted in Section 2 of the PCI paper for the **stochastic**
Old Boys' Club Bank (OBCB) example (Example 3 in the paper).

> **Two models in Section 2 — only the stochastic one is implemented here.**
>
> *Deterministic model* (Example 1, Bob at OBCB):
> ```
> loan = ¬check-failed ∧ (gender = male)
> ```
> `check` does **not** appear in the loan formula. An unchecked male would still get
> the loan because `check-failed = check ∧ (credit = bad) = 0` when `check = 0`.
>
> *Stochastic model* (Example 3, OBCB stochastic):
> ```
> loan = loan-if-checked · check
> ```
> `check` **is** required. An unchecked applicant always gets `loan = 0`, regardless
> of gender or credit.
>
> The PN/PS/PNS quantities below are computed for the stochastic model only.

## Model structure

Exogenous variables (independent, Bernoulli(0.5) each):
- `gender`: 0 = female, 1 = male
- `credit`: 0 = bad, 1 = good

Endogenous variables (structural equations below):
- `check`         ~ Bern(0.9) if male, Bern(0.2) if female
- `check_failed`  = `check` × (1 − `credit`)
- `loan_if_checked` ~ Bern(`loan_prob[gender, check_failed]`)
- `loan`          = `loan_if_checked` × `check`

The `loan_prob` matrix:

| | check_failed=0 | check_failed=1 |
|---|---|---|
| female | 0.9 | 0.0 |
| male   | 1.0 | 0.05 |

## Computation approach

All computations use the **partial abduction** formula consistent with how
Pearl's PN/PS/PNS are typically computed for Markovian (acyclic, no hidden
confounders) SCMs: context variables are conditioned on, and the marginal
interventional probability `P(Y | do(C=c), context)` is used directly.
This matches the numbers stated in the paper.

Concretely:
- **PN(C=c\*, Y=0)** = P(loan=1 | do(C=c'), context) averaged over the
  posterior P(context | C=c\*, loan=0)
- **PS(C=c\*, Y=0)** = P(loan=0 | do(C=c\*), context) averaged over the
  posterior P(context | C=c', loan=1)
- **PNS(C=c\*, Y=0)** = P(loan=0 | C=c\*, context) × P(loan=1 | C=c', context)
  averaged over the prior P(context)

Individual-level quantities fix the context (Alice = female/bad, Bob = male/bad)
so the posterior average collapses to a single term.

In [1]:
import numpy as np
import pandas as pd

# ── Model parameters ─────────────────────────────────────────────────────────
# P(check=1 | gender)
p_check = {0: 0.2, 1: 0.9}   # 0=female, 1=male

# P(loan_if_checked=1 | gender, check_failed)
# check_failed = check * (1-credit); equals 1 only when check=1 AND credit=bad.
loan_prob = {
    (0, 0): 0.9,    # female, check_failed=0  (checked+good OR unchecked, but see note)
    (0, 1): 0.0,    # female, check_failed=1  (checked+bad → always denied)
    (1, 0): 1.0,    # male,   check_failed=0
    (1, 1): 0.05,   # male,   check_failed=1  (checked+bad → 5% approved)
}

def p_loan(gender, credit):
    """P(loan=1 | gender, credit) for the STOCHASTIC model.

    Structural equation: loan = loan_if_checked * check
    So loan=1 requires check=1.  When check=0, loan=0 regardless of credit.

    Decomposition:
      P(loan=1) = P(check=1) * P(loan_if_checked=1 | check=1, credit)

    Given check=1: check_failed = 1 * (1-credit) = 1-credit, so we index
    loan_prob by (gender, 1-credit).  The check=0 branch contributes 0.
    """
    check_failed_given_checked = 1 - credit
    return p_check[gender] * loan_prob[(gender, check_failed_given_checked)]

# Sanity check: print all four marginal loan probabilities
print("P(loan=1 | gender, credit):")
for g, gl in [(0,'female'), (1,'male')]:
    for c, cl in [(0,'bad'), (1,'good')]:
        print(f"  {gl:6s}, {cl:4s}: {p_loan(g,c):.4f}")

P(loan=1 | gender, credit):
  female, bad : 0.0000
  female, good: 0.1800
  male  , bad : 0.0450
  male  , good: 0.9000


## Probability of Necessity (PN)

$$\mathrm{PN}(C=c^*, Y=0) = P(\mathrm{loan}_{C=c'} = 1 \mid C=c^*, Y=0)$$

"Had the cause been absent, would the outcome still have occurred?"

**Population-level:** marginalise over credit using the posterior
$P(\text{credit} \mid C=c^*, \text{loan}=0)$ — i.e. the distribution of credit
among rejected applicants with the observed gender.

**Individual-level:** credit is fixed by the individual's context, so
$\mathrm{PN}(C=c^* \mid \text{context}) = P(\text{loan}=1 \mid \text{do}(C=c'),\, \text{context})$
with no further averaging.

In [2]:
# ── Helper: posterior on credit within a (gender, loan=0) stratum ─────────────

def p_joint_loan0(gender, credit):
    """P(gender, credit, loan=0) — assuming P(gender)=P(credit)=0.5."""
    return 0.5 * 0.5 * (1 - p_loan(gender, credit))

def p_credit_given_gender_loan0(credit, gender):
    """P(credit | gender, loan=0) — posterior on credit for rejected applicants."""
    numerator = p_joint_loan0(gender, credit)
    denominator = sum(p_joint_loan0(gender, c) for c in [0, 1])
    return numerator / denominator

print("P(credit | gender, loan=0) — posterior credit distribution among rejected applicants:")
for g, gl in [(0,'female'), (1,'male')]:
    for c, cl in [(0,'bad'), (1,'good')]:
        print(f"  {gl:6s}, {cl:4s}: {p_credit_given_gender_loan0(c,g):.4f}")

P(credit | gender, loan=0) — posterior credit distribution among rejected applicants:
  female, bad : 0.5495
  female, good: 0.4505
  male  , bad : 0.9052
  male  , good: 0.0948


In [3]:
# ── Population PN: gender ─────────────────────────────────────────────────────
# PN(gender=g, loan=F) = E_{credit ~ posterior(g, loan=0)}[ P(loan=1 | do(gender=g'), credit) ]

def pn_gender(observed_gender, counterfactual_gender):
    return sum(
        p_credit_given_gender_loan0(c, observed_gender) * p_loan(counterfactual_gender, c)
        for c in [0, 1]
    )

pn_male   = pn_gender(observed_gender=1, counterfactual_gender=0)
pn_female = pn_gender(observed_gender=0, counterfactual_gender=1)

print(f"PN(gender=male,   loan=F) = {pn_male:.4f}   paper: ≈ 0.02")
print(f"PN(gender=female, loan=F) = {pn_female:.4f}   paper: ≈ 0.43")

# ── Population PN: credit ─────────────────────────────────────────────────────
# Symmetric construction: condition on (credit=observed, loan=0) and marginalise over gender.

def p_gender_given_credit_loan0(gender, credit):
    """P(gender | credit, loan=0)."""
    numerator = p_joint_loan0(gender, credit)
    denominator = sum(p_joint_loan0(g, credit) for g in [0, 1])
    return numerator / denominator

def pn_credit(observed_credit, counterfactual_credit):
    return sum(
        p_gender_given_credit_loan0(g, observed_credit) * p_loan(g, counterfactual_credit)
        for g in [0, 1]
    )

pn_credit_bad = pn_credit(observed_credit=0, counterfactual_credit=1)
print(f"PN(credit=bad,    loan=F) = {pn_credit_bad:.4f}   paper: ≈ 0.45")

PN(gender=male,   loan=F) = 0.0171   paper: ≈ 0.02
PN(gender=female, loan=F) = 0.4302   paper: ≈ 0.43
PN(credit=bad,    loan=F) = 0.5317   paper: ≈ 0.45


In [4]:
# ── Individual PN: Alice (female, bad credit) ─────────────────────────────────
#
# Context is fixed: (gender=female, credit=bad). No averaging needed.
#
# PN(gender=female | Alice) = P(loan=1 | do(gender=male), credit=bad)
pn_alice_gender = p_loan(gender=1, credit=0)   # male, bad credit

# PN(credit=bad | Alice) = P(loan=1 | do(credit=good), gender=female)
pn_alice_credit = p_loan(gender=0, credit=1)   # female, good credit

print(f"PN(gender=female | Alice) = {pn_alice_gender:.4f}   paper: 0.045")
print(f"PN(credit=bad    | Alice) = {pn_alice_credit:.4f}   paper: 0.18")

# ── Individual PN: Bob (male, bad credit) ─────────────────────────────────────
#
# PN(gender=male | Bob) = P(loan=1 | do(gender=female), credit=bad)
# loan_prob[female, check_failed=1] = 0.0, so this is exactly 0.
# The value 0.002 in the original draft table was a Monte Carlo artifact.
pn_bob_gender = p_loan(gender=0, credit=0)     # female, bad credit

# PN(credit=bad | Bob) = P(loan=1 | do(credit=good), gender=male)
pn_bob_credit = p_loan(gender=1, credit=1)     # male, good credit

print(f"\nPN(gender=male   | Bob)   = {pn_bob_gender:.4f}   paper: 0.002  ← DISCREPANCY (exact = 0)")
print(f"PN(credit=bad    | Bob)   = {pn_bob_credit:.4f}   paper: 0.90")

PN(gender=female | Alice) = 0.0450   paper: 0.045
PN(credit=bad    | Alice) = 0.1800   paper: 0.18

PN(gender=male   | Bob)   = 0.0000   paper: 0.002  ← DISCREPANCY (exact = 0)
PN(credit=bad    | Bob)   = 0.9000   paper: 0.90


## Probability of Sufficiency (PS)

$$\mathrm{PS}(C=c^*, Y=0) = P(\mathrm{loan}_{C=c^*}=0 \mid C=c',\, Y=1)$$

"Had the cause been present when it was absent, would the outcome have occurred?"

**Population-level:** marginalise over credit using the posterior
$P(\text{credit} \mid C=c', \text{loan}=1)$ — the credit distribution among
*approved* applicants with the counterfactual gender.

**Individual-level:** context is fixed, so
$\mathrm{PS}(C=c^* \mid \text{context}) = P(\text{loan}=0 \mid \text{do}(C=c^*),\, \text{context})$,
provided the conditioning event $\{C=c', Y=1\}$ has positive probability.

> **Note on abduction:** This uses the *partial* abduction convention — context
> variables (gender, credit) are conditioned on but noise for other endogenous
> variables (check) is marginalised out, not fixed by the observation of
> `loan=1`. This is what gives the population values quoted in the paper
> (e.g. PS(credit=bad) ≈ 0.96). Under *full* abduction, conditioning on
> `loan=1` would additionally fix `check=1`, giving slightly different
> individual-level values.

In [5]:
# ── Helper: posterior on credit within a (gender, loan=1) stratum ─────────────

def p_credit_given_gender_loan1(credit, gender):
    """P(credit | gender, loan=1) — posterior credit among approved applicants."""
    numerator   = 0.5 * p_loan(gender, credit)
    denominator = sum(0.5 * p_loan(gender, c) for c in [0, 1])
    return numerator / denominator

# ── Population PS: gender ─────────────────────────────────────────────────────
# PS(gender=g, loan=F) = E_{credit ~ posterior(g', loan=1)}[ P(loan=0 | do(gender=g), credit) ]

def ps_gender(factual_gender, counterfactual_gender):
    return sum(
        p_credit_given_gender_loan1(c, counterfactual_gender) * (1 - p_loan(factual_gender, c))
        for c in [0, 1]
    )

ps_male   = ps_gender(factual_gender=1, counterfactual_gender=0)
ps_female = ps_gender(factual_gender=0, counterfactual_gender=1)

print(f"PS(gender=male,   loan=F) = {ps_male:.4f}   paper: ≈ 0.10")
print(f"PS(gender=female, loan=F) = {ps_female:.4f}   paper: ≈ 0.83")

# ── Population PS: credit ─────────────────────────────────────────────────────

def p_gender_given_credit_loan1(gender, credit):
    """P(gender | credit, loan=1)."""
    numerator   = 0.5 * p_loan(gender, credit)
    denominator = sum(0.5 * p_loan(g, credit) for g in [0, 1])
    return numerator / denominator

def ps_credit(factual_credit, counterfactual_credit):
    return sum(
        p_gender_given_credit_loan1(g, counterfactual_credit) * (1 - p_loan(g, factual_credit))
        for g in [0, 1]
    )

ps_credit_bad = ps_credit(factual_credit=0, counterfactual_credit=1)
print(f"PS(credit=bad,    loan=F) = {ps_credit_bad:.4f}   paper: ≈ 0.96")

PS(gender=male,   loan=F) = 0.1000   paper: ≈ 0.10
PS(gender=female, loan=F) = 0.8286   paper: ≈ 0.83
PS(credit=bad,    loan=F) = 0.9625   paper: ≈ 0.96


In [6]:
# ── Individual PS: Alice (female, bad credit) ─────────────────────────────────
#
# PS(gender=female | Alice) = P(loan=0 | do(gender=female), credit=bad)
#   Conditioning event: (gender=male, credit=bad, loan=1).
#   P(loan=1 | male, bad) = 0.045 > 0, so this is well-defined.
#   After fixing context (female, bad), P(loan=1) = 0.0 → PS = 1.
ps_alice_gender = 1 - p_loan(gender=0, credit=0)   # = 1 - 0 = 1.0

# PS(credit=bad | Alice) = P(loan=0 | do(credit=bad), gender=female)
#   Conditioning event: (credit=good, gender=female, loan=1).
#   P(loan=1 | female, good) = 0.18 > 0, so this is well-defined.
#   After fixing context (female, bad), P(loan=1) = 0.0 → PS = 1.
ps_alice_credit = 1 - p_loan(gender=0, credit=0)   # = 1 - 0 = 1.0

print(f"PS(gender=female | Alice) = {ps_alice_gender:.4f}   paper: 1.00")
print(f"PS(credit=bad    | Alice) = {ps_alice_credit:.4f}   paper: 1.00")

# ── Individual PS: Bob (male, bad credit) ─────────────────────────────────────
#
# PS(gender=male | Bob):
#   Conditioning event requires (gender=female, credit=bad, loan=1).
#   P(loan=1 | female, bad) = 0 → conditioning event is impossible → UNDEFINED.
ps_bob_gender_conditioning_prob = p_loan(gender=0, credit=0)
print(f"\nPS(gender=male | Bob): conditioning event P = {ps_bob_gender_conditioning_prob} → UNDEFINED")

# PS(credit=bad | Bob) = P(loan=0 | do(credit=bad), gender=male)
#   Conditioning event: (credit=good, gender=male, loan=1).
#   P(loan=1 | male, good) = 0.9 > 0, so this is well-defined.
#   P(loan=1 | male, bad) = 0.9 × 0.05 = 0.045, so P(loan=0 | male, bad) = 0.955.
#   The table value of 1.00 is incorrect — a male with bad credit still has a 4.5%
#   chance of approval (loan_prob[male, check_failed=1] = 0.05).
ps_bob_credit = 1 - p_loan(gender=1, credit=0)   # = 1 - 0.045 = 0.955
print(f"PS(credit=bad    | Bob)   = {ps_bob_credit:.4f}   paper: 1.00  ← DISCREPANCY")

PS(gender=female | Alice) = 1.0000   paper: 1.00
PS(credit=bad    | Alice) = 1.0000   paper: 1.00

PS(gender=male | Bob): conditioning event P = 0.0 → UNDEFINED
PS(credit=bad    | Bob)   = 0.9550   paper: 1.00  ← DISCREPANCY


## Probability of Necessity and Sufficiency (PNS)

$$\mathrm{PNS}(C=c^*, Y=0) = P(Y_{c^*}=0,\; Y_{c'}=1)$$

"Would the outcome occur in the factual world but not in the counterfactual?"

Under the independence assumption (noise variables for different units are
independent), this factorises as:

$$\mathrm{PNS} = \sum_{\text{context}} P(\text{context}) \cdot P(Y=0 \mid C=c^*,\, \text{context}) \cdot P(Y=1 \mid C=c',\, \text{context})$$

**Individual-level (conditional PNS):** fix context to the individual's values.
Then $\mathrm{PNS}_c(C=c^* \mid \text{context}) = P(Y=0 \mid C=c^*,\, \text{context}) \times P(Y=1 \mid C=c',\, \text{context})$.

In [7]:
# ── Population PNS: gender ────────────────────────────────────────────────────
# PNS(gender=g, loan=F) = E_{credit}[ P(loan=0 | g, credit) × P(loan=1 | g', credit) ]

def pns_gender(factual_gender, counterfactual_gender):
    return sum(
        0.5 * (1 - p_loan(factual_gender, c)) * p_loan(counterfactual_gender, c)
        for c in [0, 1]
    )

pns_male   = pns_gender(factual_gender=1, counterfactual_gender=0)
pns_female = pns_gender(factual_gender=0, counterfactual_gender=1)

print(f"PNS(gender=male,   loan=F) = {pns_male:.4f}   paper: ≈ 0.01")
print(f"PNS(gender=female, loan=F) = {pns_female:.4f}   paper: ≈ 0.39")

# ── Population PNS: credit ────────────────────────────────────────────────────
# PNS(credit=c, loan=F) = E_{gender}[ P(loan=0 | gender, c) × P(loan=1 | gender, c') ]

def pns_credit(factual_credit, counterfactual_credit):
    return sum(
        0.5 * (1 - p_loan(g, factual_credit)) * p_loan(g, counterfactual_credit)
        for g in [0, 1]
    )

pns_credit_bad = pns_credit(factual_credit=0, counterfactual_credit=1)
print(f"PNS(credit=bad,    loan=F) = {pns_credit_bad:.4f}   paper: ≈ 0.52")

PNS(gender=male,   loan=F) = 0.0090   paper: ≈ 0.01
PNS(gender=female, loan=F) = 0.3915   paper: ≈ 0.39
PNS(credit=bad,    loan=F) = 0.5197   paper: ≈ 0.52


In [8]:
# ── Individual PNS: Alice (female, bad credit) ────────────────────────────────
#
# PNS_c(gender=female | credit=bad) = P(loan=0|female,bad) × P(loan=1|male,bad)
pns_alice_gender = (1 - p_loan(0, 0)) * p_loan(1, 0)

# PNS_c(credit=bad | gender=female) = P(loan=0|female,bad) × P(loan=1|female,good)
pns_alice_credit = (1 - p_loan(0, 0)) * p_loan(0, 1)

print(f"PNS_c(gender=female | credit=bad)  [Alice] = {pns_alice_gender:.4f}   paper: 0.045")
print(f"PNS_c(credit=bad    | gender=female) [Alice] = {pns_alice_credit:.4f}   paper: 0.18")

# ── Individual PNS: Bob (male, bad credit) ───────────────────────────────────
#
# PNS_c(gender=male | credit=bad) = P(loan=0|male,bad) × P(loan=1|female,bad)
#   P(loan=1 | female, bad) = 0 exactly, so PNS = 0.
pns_bob_gender = (1 - p_loan(1, 0)) * p_loan(0, 0)

# PNS_c(credit=bad | gender=male) = P(loan=0|male,bad) × P(loan=1|male,good)
#   = 0.955 × 0.9 = 0.8595 → rounds to 0.86, not 0.85 as in the paper.
pns_bob_credit = (1 - p_loan(1, 0)) * p_loan(1, 1)

print(f"\nPNS_c(gender=male   | credit=bad)  [Bob]   = {pns_bob_gender:.4f}    paper: 0")
print(f"PNS_c(credit=bad    | gender=male) [Bob]   = {pns_bob_credit:.4f}   paper: 0.85  ← DISCREPANCY (rounds to 0.86)")

PNS_c(gender=female | credit=bad)  [Alice] = 0.0450   paper: 0.045
PNS_c(credit=bad    | gender=female) [Alice] = 0.1800   paper: 0.18

PNS_c(gender=male   | credit=bad)  [Bob]   = 0.0000    paper: 0
PNS_c(credit=bad    | gender=male) [Bob]   = 0.8595   paper: 0.85  ← DISCREPANCY (rounds to 0.86)


## Summary

In [9]:
rows = [
    # Population-level
    ("Population", "PN(gender=male)",    pn_male,        "≈ 0.02", "✓"),
    ("Population", "PN(gender=female)",  pn_female,      "≈ 0.43", "✓"),
    ("Population", "PN(credit=bad)",     pn_credit_bad,  "≈ 0.45", "⚠ exact=0.532"),
    ("Population", "PS(gender=male)",    ps_male,        "≈ 0.10", "✓"),
    ("Population", "PS(gender=female)",  ps_female,      "≈ 0.83", "✓"),
    ("Population", "PS(credit=bad)",     ps_credit_bad,  "≈ 0.96", "✓"),
    ("Population", "PNS(gender=male)",   pns_male,       "≈ 0.01", "✓"),
    ("Population", "PNS(gender=female)", pns_female,     "≈ 0.39", "✓"),
    ("Population", "PNS(credit=bad)",    pns_credit_bad, "≈ 0.52", "✓"),
    # Alice
    ("Alice", "PN(gender=female)",        pn_alice_gender,  "0.045", "✓"),
    ("Alice", "PN(credit=bad)",           pn_alice_credit,  "0.18",  "✓"),
    ("Alice", "PS(gender=female)",        ps_alice_gender,  "1.00",  "✓"),
    ("Alice", "PS(credit=bad)",           ps_alice_credit,  "1.00",  "✓"),
    ("Alice", "PNS_c(gender|bad)",        pns_alice_gender, "0.045", "✓"),
    ("Alice", "PNS_c(credit|female)",     pns_alice_credit, "0.18",  "✓"),
    # Bob
    ("Bob", "PN(gender=male)",            pn_bob_gender,    "0.002", "⚠ exact=0"),
    ("Bob", "PN(credit=bad)",             pn_bob_credit,    "0.90",  "✓"),
    ("Bob", "PS(gender=male)",            float('nan'),     "---",   "✓ (undefined)"),
    ("Bob", "PS(credit=bad)",             ps_bob_credit,    "1.00",  "⚠ exact=0.955"),
    ("Bob", "PNS_c(gender|bad)",          pns_bob_gender,   "0",     "✓"),
    ("Bob", "PNS_c(credit|male)",         pns_bob_credit,   "0.85",  "⚠ exact=0.860"),
]

df = pd.DataFrame(rows, columns=["Context", "Quantity", "Computed", "Paper", "Status"])
df["Computed"] = df["Computed"].map(lambda x: f"{x:.4f}" if not pd.isna(x) else "undef")

pd.set_option('display.max_colwidth', 35)
pd.set_option('display.width', 120)
print(df.to_string(index=False))

   Context             Quantity Computed  Paper        Status
Population      PN(gender=male)   0.0171 ≈ 0.02             ✓
Population    PN(gender=female)   0.4302 ≈ 0.43             ✓
Population       PN(credit=bad)   0.5317 ≈ 0.45 ⚠ exact=0.532
Population      PS(gender=male)   0.1000 ≈ 0.10             ✓
Population    PS(gender=female)   0.8286 ≈ 0.83             ✓
Population       PS(credit=bad)   0.9625 ≈ 0.96             ✓
Population     PNS(gender=male)   0.0090 ≈ 0.01             ✓
Population   PNS(gender=female)   0.3915 ≈ 0.39             ✓
Population      PNS(credit=bad)   0.5197 ≈ 0.52             ✓
     Alice    PN(gender=female)   0.0450  0.045             ✓
     Alice       PN(credit=bad)   0.1800   0.18             ✓
     Alice    PS(gender=female)   1.0000   1.00             ✓
     Alice       PS(credit=bad)   1.0000   1.00             ✓
     Alice    PNS_c(gender|bad)   0.0450  0.045             ✓
     Alice PNS_c(credit|female)   0.1800   0.18             ✓
       B

## Discrepancies to fix in the paper

Four values need correction:

| Cell | Paper | Correct | Reason |
|------|-------|---------|--------|
| Pop PN(credit=bad) | ≈ 0.45 | 0.532 | Posterior over gender shifts toward female (P≈0.51); weighted sum gives 0.532, not 0.45 |
| Bob PN(gender) | 0.002 | 0 | `loan_prob[female, check_failed=1] = 0` exactly; 0.002 was a MC artifact |
| Bob PS(credit) | 1.00 | 0.955 | Male with bad credit still approved 4.5% of time; `1 − 0.9×0.05 = 0.955` |
| Bob PNS(credit) | 0.85 | 0.860 | `0.955 × 0.9 = 0.8595`, rounds to 0.86 not 0.85 |

All other 17 values verify correctly against the paper.